# 第 73 课：真正的 KV Cache、分页块表与生命周期审计

上一课把请求调度拆成 prefill/decode；其小模型为便于校验，decode 每一步都重新计算完整前缀。**这一课实现真正的增量 K/V 张量缓存**，并为它补齐显存账本、物理块分配、共享前缀、写时复制、OOM 回滚与终态清理。

本课包括真实 CPU PyTorch 因果模型（两个只读 Q/V LoRA 版本）和确定性资源调度夹具。**缓存背后确实有物理 Tensor，前向确实读取之前保存的 K/V**；但 block table 和逐层读取以 Python 实现，**并不是 vLLM 的 PagedAttention GPU 融合 Kernel，也不是吞吐测试**。

推荐完成顺序：先核对不带缓存与带缓存的数值等价 → 容量公式 → fork/COW → 生命周期 → 自动检查 → 三道独立练习。

## 0. 运行方式和依赖

使用已有 PyTorch 和 Jupyter 环境，从全新内核按顺序运行。这份 Notebook 完整独立，不导入上一课脚本，不下载权重、数据或分词器。

配套命令行：
```bash
python lesson_73_kv_cache_paged_blocks_lifecycle.py --checks-only
python lesson_73_kv_cache_paged_blocks_lifecycle.py --demo-only
python lesson_73_kv_cache_paged_blocks_lifecycle.py
python lesson_73_kv_cache_paged_blocks_lifecycle.py --reload-run lesson_73_outputs/实际运行目录
```

默认完整演练开启，输出到 `lesson_73_outputs/run_时间戳/`，每次运行新建目录，不覆盖之前文件。如果暂时只看数学与检查，把下面的 `RUN_FULL_DEMO` 改为 False。

**重要：** 本课的内存值分为预分配的 Tensor 存储大小、已分配的存活物理块字节和真正有效 token 的逻辑字节。它们不等于完整进程 RSS/GPU 显存。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第73课：真正的增量KV缓存、分页块表、写时复制与全生命周期回收。

独立离线PyTorch教学实验。真实微型因果Transformer/LoRA在物理KV张量里增量读写，
与无缓存全上下文前向核对；纯CPU分页管理与服务事件为模型机制演示。
没有PagedAttention融合GPU内核、真实GPU性能数据或生产级服务并发。
"""
from __future__ import annotations

# %% 01 环境、确定性身份与范围
import argparse
import copy
import hashlib
import json
import math
import platform
import random
import re
import tempfile
from collections import Counter
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit("需要已有PyTorch环境；本课不自动安装或下载模型。") from exc

VERSION = "lesson73-kv-pages-v1"
SOURCES = {
    "pagedattention": "https://arxiv.org/abs/2309.06180",
    "transformers_cache": "https://huggingface.co/docs/transformers/main/en/cache_explanation",
    "pytorch_attention": "https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html",
    "pytorch_inference": "https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html",
    "reproducibility": "https://docs.pytorch.org/docs/stable/notes/randomness.html",
}

class LessonError(ValueError):
    """拒绝含糊的token边界、停止状态、报告格式或资源预算。"""

def require(ok: bool, msg: str) -> None:
    if not bool(ok):
        raise LessonError(msg)

def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)

def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()

def write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True,
                               allow_nan=False) + "\n", encoding="utf-8")

def bounded_str(value: Any, length: int, label: str) -> str:
    require(type(value) is str and len(value.encode("utf-8")) <= length,
            label + "必须是有界字符串")
    return value

def seed_all(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def state_hash(state: dict[str,torch.Tensor]) -> str:
    h=hashlib.sha256()
    for key,tensor in sorted(state.items()):
        t=tensor.detach().cpu().contiguous()
        h.update(canonical([key,str(t.dtype),list(t.shape)]).encode())
        h.update(t.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

In [2]:
torch.set_num_threads(1)
RUN_FULL_DEMO = True
OUTPUT_ROOT = Path("lesson_73_outputs")
SEED = 7201
seed_all(SEED)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
print("默认仅CPU FP32，真实GPU KV内核/显存未测试。")

Python: 3.13.5
PyTorch: 2.10.0+cpu CUDA available: False
默认仅CPU FP32，真实GPU KV内核/显存未测试。


## 1. KV Cache 实际保存什么？

在因果自注意力的某层，每个输入 token 在该层生成 `K` 和 `V`。已经处理过的 token 不受未来 token 的影响，所以自回归推理可以缓存这些层的历史 K/V。

设模型有 `L` 层、每层 `H_kv` 个 K/V 头、每头 `D_head` 维、dtype 每元素 `b` 字节：

\[
\text{KV bytes per token}=2\times L\times H_{kv}\times D_{head}\times b
\]

前面的 2 表示 **K 和 V**，不是 query 或 batch size。使用 GQA 时按 K/V 头数算，不是按 query 头数算。注意这里**没有计算模型权重、注意力临时张量、分词器、CUDA分配器或 Python 对象**。

参考：[Hugging Face KV Cache 解释](https://huggingface.co/docs/transformers/main/en/cache_explanation) · [PagedAttention 论文](https://arxiv.org/abs/2309.06180)。

## 2. 先保留一个可核对的因果语言模型

教学模型：二维结构完整的两层因果 Transformer、绝对位置、LayerNorm、Q/K/V/O 和 FFN；再分别注入两份 Q/V LoRA。两个版本基座相同，但 adapter 不同，因此**不能共享同一份物理 K/V**。

分词器沿用第 72 课的教学概念：控制 token ID 与普通文本字节分开，避免把输入中的 `<im_start>` 字面文本误识别为结构符。它不是 Hugging Face 真实 tokenizer，迁移到 Qwen 应使用模型官方 tokenizer/模板与 cache API。

这部分代码较长，但内嵌在本文件里，零外部源码依赖。

In [3]:
# %% 02 固定教学分词器与真实因果/LoRA模型（独立于前课文件）
class ByteMergeChatTokenizer:
    """自建教学分词器：控制token独立于用户文本，普通文本按UTF-8字节合并。

    不假装实现HF的FastTokenizer或任意模型Chat Template。固定控制ID，避免把用户
    输入里的字面控制符误当结构token。真实应用必须使用对应模型的官方tokenizer/template。
    """
    pad_token_id = 0
    bos_token_id = 1
    eos_token_id = 2
    im_start_token_id = 3
    im_end_token_id = 4
    role_ids = {"system":5, "user":6, "assistant":7}
    byte_offset = 16
    merges = (b"OK", b" 42", b"42", b"hello", b"world", b"\n", b"answer", b"JSON", b"{\"answer\":")

    def __init__(self):
        self.merge_ids = {piece: self.byte_offset + 256 + i for i,piece in enumerate(self.merges)}
        self.ids_to_merge = {v:k for k,v in self.merge_ids.items()}
        self.vocab_size = self.byte_offset + 256 + len(self.merges)
        self.specials = {0:"<pad>",1:"<bos>",2:"<eos>",3:"<im_start>",
                         4:"<im_end>",5:"<system>",6:"<user>",7:"<assistant>"}

    def encode(self, text: str, *, add_special_tokens: bool=False) -> list[int]:
        bounded_str(text, 4096, "文本")
        raw = text.encode("utf-8")
        result = []
        index = 0
        while index < len(raw):
            matches = [piece for piece in self.merges if raw.startswith(piece, index)]
            if matches:
                piece = max(matches, key=len)
                result.append(self.merge_ids[piece]);index += len(piece)
            else:
                result.append(self.byte_offset + raw[index]);index += 1
        return ([self.bos_token_id] if add_special_tokens else []) + result

    def decode(self, ids: list[int], *, skip_special_tokens: bool=False) -> str:
        require(type(ids) is list, "token IDs必须是列表")
        raw = bytearray()
        for i in ids:
            require(type(i) is int and 0 <= i < self.vocab_size, "token ID越界")
            if i in self.specials:
                if skip_special_tokens:
                    continue
                raise LessonError("正文decode不允许控制token；传入skip_special_tokens=True")
            if i in self.ids_to_merge: raw.extend(self.ids_to_merge[i])
            elif self.byte_offset <= i < self.byte_offset + 256: raw.append(i - self.byte_offset)
            else: raise LessonError("保留ID不是合法正文token")
        try:
            return bytes(raw).decode("utf-8",errors="strict")
        except UnicodeDecodeError as exc:
            raise LessonError("不完整UTF-8回答；不要用replacement字符伪装成功") from exc

    def apply_chat_template(self, messages: list[dict], *, tokenize: bool=True,
                            add_generation_prompt: bool=False):
        require(type(messages) is list and bool(messages), "messages需要非空列表")
        require(type(add_generation_prompt) is bool, "generation prompt必须是bool")
        ids = [self.bos_token_id]
        for j,m in enumerate(messages):
            require(type(m) is dict and set(m)=={"role","content"}, "message字段必须恰好role/content")
            require(m["role"] in self.role_ids and type(m["content"]) is str, "角色或内容非法")
            require(not (j == len(messages)-1 and add_generation_prompt and m["role"]=="assistant"),
                    "若最后已有assistant，请使用续写语义，不要再添加generation prompt")
            ids.extend([self.im_start_token_id,self.role_ids[m["role"]]])
            ids.extend(self.encode(m["content"],add_special_tokens=False))
            ids.append(self.eos_token_id if m["role"]=="assistant" else self.im_end_token_id)
        if add_generation_prompt:
            ids.extend([self.im_start_token_id,self.role_ids["assistant"]])
        if tokenize:
            return ids
        # 仅用于人类阅读，控制符是标记，不要将其当普通文本用encode()重新编码。
        visible = []
        for i in ids:
            if i in self.specials: visible.append(self.specials[i])
            else: visible.append(self.decode([i],skip_special_tokens=True) if
                                 self._token_is_full_utf8(i) else "<byte-part>")
        return "".join(visible)

    def _token_is_full_utf8(self, i):
        if i in self.ids_to_merge:
            try: self.ids_to_merge[i].decode("utf-8");return True
            except UnicodeError: return False
        return self.byte_offset <= i < self.byte_offset + 128

    def contract(self) -> dict:
        return {"type":"toy-byte-merge", "specials":self.specials,
                "merges":[p.hex() for p in self.merges],"vocab":self.vocab_size,
                "prefix":"BOS+message start/role/body+end+assistant header",
                "eos":self.eos_token_id,"pad":self.pad_token_id}

@dataclass(frozen=True)
class ToyConfig:
    vocab_size: int
    dim: int=32
    heads: int=4
    layers: int=2
    ff_dim: int=64
    max_length: int=160

    def validate(self):
        require(all(type(x) is int and x>0 for x in asdict(self).values()),'配置必须为正整数')
        require(self.dim%self.heads==0 and self.vocab_size>16,
                'dim/head或词表不匹配')

class CausalSelfAttention(nn.Module):
    def __init__(self,cfg: ToyConfig):
        super().__init__()
        d=cfg.dim
        self.heads=cfg.heads
        self.q_proj=nn.Linear(d,d,bias=False)
        self.k_proj=nn.Linear(d,d,bias=False)
        self.v_proj=nn.Linear(d,d,bias=False)
        self.o_proj=nn.Linear(d,d,bias=False)

    def forward(self,x:torch.Tensor,attention_mask:torch.Tensor):
        b,t,d=x.shape;h=self.heads
        def split(layer):
            return layer(x).reshape(b,t,h,d//h).transpose(1,2)
        q,k,v=split(self.q_proj),split(self.k_proj),split(self.v_proj)
        scores=(q@k.transpose(-1,-2))/math.sqrt(d//h)
        lower=torch.ones(t,t,device=x.device,dtype=torch.bool).tril()
        allowed=lower[None,None,:,:]&attention_mask[:,None,None,:]
        weights=torch.softmax(scores.masked_fill(~allowed,float('-inf')),dim=-1)
        result=(weights@v).transpose(1,2).contiguous().reshape(b,t,d)
        return self.o_proj(result)

class TransformerBlock(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__()
        self.norm1=nn.LayerNorm(cfg.dim)
        self.attn=CausalSelfAttention(cfg)
        self.norm2=nn.LayerNorm(cfg.dim)
        self.ff=nn.Sequential(nn.Linear(cfg.dim,cfg.ff_dim),nn.GELU(),
                              nn.Linear(cfg.ff_dim,cfg.dim))

    def forward(self,x,mask):
        x=x+self.attn(self.norm1(x),mask)
        return x+self.ff(self.norm2(x))

class TinyChatLM(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__();cfg.validate();self.cfg=cfg
        self.embed=nn.Embedding(cfg.vocab_size,cfg.dim)
        self.position=nn.Embedding(cfg.max_length,cfg.dim)
        self.blocks=nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.layers)])
        self.norm=nn.LayerNorm(cfg.dim)
        self.head=nn.Linear(cfg.dim,cfg.vocab_size,bias=False)

    def forward(self,input_ids:torch.Tensor,attention_mask:torch.Tensor):
        require(input_ids.ndim==2 and input_ids.dtype==torch.long,'输入必须为[B,T] long')
        b,t=input_ids.shape
        require(b>0 and t>=1 and t<=self.cfg.max_length,'形状或上下文超限')
        require(attention_mask.shape==input_ids.shape and attention_mask.dtype==torch.bool,
                'attention mask形状/type不符')
        require(bool(attention_mask[:,0].all()),'不接受左补齐/空提示')
        require(not bool(((~attention_mask[:,:-1])&attention_mask[:,1:]).any()),
                '只允许右补齐')
        positions=torch.arange(t,device=input_ids.device)
        x=self.embed(input_ids)+self.position(positions)[None]
        for layer in self.blocks:x=layer(x,attention_mask)
        return self.head(self.norm(x))

class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,rank=4,alpha=8.0):
        super().__init__()
        require(type(base) is nn.Linear and type(rank) is int and rank>0,
                '手写LoRA仅支持nn.Linear与正整数rank')
        require(type(alpha) in (int,float) and math.isfinite(alpha) and alpha>0,
                'LoRA alpha非法')
        self.base=base
        self.base.requires_grad_(False)
        self.rank=rank;self.alpha=float(alpha);self.enabled=True
        self.A=nn.Parameter(base.weight.new_empty(rank,base.in_features))
        self.B=nn.Parameter(base.weight.new_zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

    def forward(self,x):
        result=self.base(x)
        if not self.enabled:return result
        return result+(self.alpha/self.rank)*F.linear(F.linear(x,self.A),self.B)

def inject_lora(base:TinyChatLM,rank=4,alpha=8.0) -> TinyChatLM:
    require(not any(isinstance(m,LoRALinear) for m in base.modules()),
            '不允许重复注入LoRA')
    policy=copy.deepcopy(base)
    policy.requires_grad_(False)
    for block in policy.blocks:
        block.attn.q_proj=LoRALinear(block.attn.q_proj,rank,alpha)
        block.attn.v_proj=LoRALinear(block.attn.v_proj,rank,alpha)
    return policy

def adapters(model:TinyChatLM):
    return [(n,p) for n,p in model.named_parameters() if n.endswith(('.A','.B'))]

def adapter_snapshot(model:TinyChatLM):
    return {n:p.detach().cpu().clone() for n,p in adapters(model)}

def restore_adapter(model:TinyChatLM,values:dict):
    parameters=dict(model.named_parameters())
    require(set(values)==set(adapter_snapshot(model)),'adapter键集合不匹配')
    with torch.no_grad():
        for name,v in values.items():
            require(v.shape==parameters[name].shape and bool(torch.isfinite(v).all()),
                    'adapter参数形状/数值不匹配')
            parameters[name].copy_(v.to(parameters[name]))

def base_state(model:TinyChatLM):
    base={}
    for name,value in model.state_dict().items():
        if name.endswith(('.A','.B')):continue
        normalized=name.replace('.q_proj.base.','.q_proj.').replace('.v_proj.base.','.v_proj.')
        require(normalized not in base,'归一化基座命名冲突')
        base[normalized]=value
    return base

def base_hash(model:TinyChatLM):
    return state_hash(base_state(model))

In [4]:
# %% 03 同基座不同Adapter的只读模型版本
def build_models(seed: int=7201):
    """同一随机基座 + 两份独立Q/V LoRA。版本只是不可变教学对象。"""
    require(type(seed) is int and seed >= 0, 'seed非法')
    seed_all(seed)
    tok = ByteMergeChatTokenizer()
    cfg = ToyConfig(tok.vocab_size, dim=32, heads=4, layers=2, ff_dim=64, max_length=192)
    base = TinyChatLM(cfg)
    a = inject_lora(base, rank=4, alpha=8.0)
    b = inject_lora(base, rank=4, alpha=8.0)
    # B=0时LoRA作用为零；通过不同adapter B制造第二个可核对的版本。
    generator = torch.Generator().manual_seed(seed+1)
    with torch.no_grad():
        for _,param in adapters(b):
            if param is not None and param.ndim == 2 and param.shape[1] == 4:  # B:[out,rank]
                param.copy_(torch.randn(param.shape, generator=generator)*0.12)
    models={'v0001':a.requires_grad_(False).eval(),
            'v0002':b.requires_grad_(False).eval()}
    require(all(not p.requires_grad for m in models.values() for p in m.parameters()),
            '服务模型必须全部冻结')
    require(base_hash(models['v0001']) == base_hash(models['v0002']),
            '两个版本必须来自同一基座')
    require(state_hash(models['v0001'].state_dict()) != state_hash(models['v0002'].state_dict()),
            '版本权重应该不同')
    return tok,models

In [5]:
tokenizer, demo_models = build_models(SEED)
print("版本:", sorted(demo_models))
print("tokenizer字节词表大小:", tokenizer.vocab_size)
print("层/头/每头维度:", demo_models['v0001'].cfg.layers, demo_models['v0001'].cfg.heads,
      demo_models['v0001'].cfg.dim//demo_models['v0001'].cfg.heads)
print("两个版本基座相同:", base_hash(demo_models['v0001']) == base_hash(demo_models['v0002']))
print("完整权重身份不同:", state_hash(demo_models['v0001'].state_dict()) !=
      state_hash(demo_models['v0002'].state_dict()))

版本: ['v0001', 'v0002']
tokenizer字节词表大小: 281
层/头/每头维度: 2 4 8
两个版本基座相同: True
完整权重身份不同: True


## 3. 真正的物理 KV 页池：分配前先确认内存

本课用一个 CPU Tensor 承载所有层的 K/V：

```text
physical_storage[block, layer, K_or_V, slot, kv_head, head_dim]
```

每个会话持有一个**逻辑块表**，指向若干不必相邻的物理 block ID。每个 block 容纳 `block_size` 个 token 的 K/V。

```text
会话A：tokens 0..9   → [物理块 5, 物理块 1, 物理块 8]
                         │            │            │
块表顺序决定逻辑上下文顺序；物理ID可以不连续。
```

预先分配的 backing Tensor 不会随会话清空而缩小。`physical_storage_bytes` 是页池预留；`live_reserved_bytes` 是当前已从 free list 领取的页；`live_content_bytes` 只计物理页有效槽。三个数字必须分开。

这与 PagedAttention 的分页思想相通，但**没有实现 fused paged attention kernel、分页换入换出、远程KV传输或复杂调度策略**。

In [6]:
# %% 04 实体KV页池：容量、块表、引用计数与回滚
@dataclass
class KVSequence:
    version: str
    tokens: list[int] = field(default_factory=list)
    block_table: list[int] = field(default_factory=list)
    pending: dict | None = None


class NoKVSpace(LessonError):
    """资源耗尽是明确失败，不能当成EOS、正常输出或偷偷丢弃KV。"""


class PagedKVPool:
    """实际CPU Tensor承载所有layer的K/V；块表是教学页索引而非vLLM kernel。

    backing: [block, layer, key_or_value, slot, kv_head, head_dim]。
    模型和版本不可在已有序列上切换；共享块只在相同版本下合法。
    尾块写入时若被共享，先Copy-on-Write复制整个块所有层再写新token。
    """
    def __init__(self, capacity_blocks: int, block_size: int, model_cfg: ToyConfig,
                 versions: set[str], dtype: torch.dtype=torch.float32):
        require(type(capacity_blocks) is int and capacity_blocks>0,'块容量必须是正整数')
        require(type(block_size) is int and block_size>0,'块token大小必须是正整数')
        model_cfg.validate()
        require(type(versions) is set and bool(versions) and all(type(v) is str and v for v in versions),
                '版本集合非法')
        require(dtype in {torch.float32,torch.float64},'教学池仅实现float32/float64')
        self.cfg=model_cfg;self.capacity_blocks=capacity_blocks;self.block_size=block_size
        self.versions=set(versions);self.dtype=dtype
        self.head_dim=model_cfg.dim//model_cfg.heads
        self.data=torch.zeros((capacity_blocks,model_cfg.layers,2,block_size,
                               model_cfg.heads,self.head_dim),dtype=dtype,device='cpu')
        self.refs=[0]*capacity_blocks
        self.free=list(range(capacity_blocks))
        self.sessions:dict[str,KVSequence]={}
        self.allocation_count=0;self.reuse_count=0;self.cow_count=0
        self.oom_count=0;self.peak_live_blocks=0;self.peak_sessions=0
        self.zeroized_releases=0

    @property
    def bytes_per_token(self):
        return 2*self.cfg.layers*self.cfg.heads*self.head_dim*self.data.element_size()

    @property
    def bytes_per_block(self):
        return self.bytes_per_token*self.block_size

    def _sid(self,sid):
        bounded_str(sid,48,'会话ID')
        require(bool(re.fullmatch(r'[A-Za-z0-9_-]+',sid)),'会话ID仅限字母数字_-')
        return sid

    def _alloc(self):
        if not self.free:
            self.oom_count+=1
            raise NoKVSpace('KV页池容量不足；拒绝本次操作且不静默覆盖缓存')
        bid=self.free.pop(0)
        require(self.refs[bid]==0,'空闲块引用数不为0')
        require(not bool(self.data[bid].count_nonzero()),'空闲块残留数据，拒绝泄漏')
        self.refs[bid]=1;self.allocation_count+=1
        self.peak_live_blocks=max(self.peak_live_blocks,self.capacity_blocks-len(self.free))
        return bid

    def _decref(self,bid):
        require(type(bid) is int and 0<=bid<self.capacity_blocks and self.refs[bid]>0,
                '引用计数下溢或块ID非法')
        self.refs[bid]-=1
        if self.refs[bid]==0:
            with torch.no_grad():self.data[bid].zero_()
            self.free.append(bid);self.free.sort();self.zeroized_releases+=1

    def open(self,sid:str,version:str):
        self._sid(sid)
        require(version in self.versions,'未知模型版本不能建缓存')
        require(sid not in self.sessions,'不能重复建同一会话')
        self.sessions[sid]=KVSequence(version=version)
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))

    def fork(self,parent:str,child:str,version:str):
        self._sid(child)
        require(parent in self.sessions and child not in self.sessions,'fork父会话缺失或子会话已存在')
        p=self.sessions[parent]
        require(p.pending is None,'不能复制正在写入的缓存')
        require(version==p.version,'跨模型版本不能共享KV：即使基座相同adapter也可能不同')
        for bid in p.block_table:self.refs[bid]+=1
        self.sessions[child]=KVSequence(version=version,tokens=p.tokens.copy(),
                                        block_table=p.block_table.copy())
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))
        self.audit()

    def begin(self,sid:str,token_id:int,max_length:int):
        require(sid in self.sessions,'不存在的会话')
        s=self.sessions[sid]
        require(s.pending is None,'同一会话不允许并发写入')
        require(type(token_id) is int and token_id>=0,'新token ID非法')
        require(type(max_length) is int and len(s.tokens)<max_length,'超过上下文长度')
        slot=len(s.tokens)%self.block_size
        new_page=False;cow_old=None
        # 先检查容量，错误时保留全部原有块表和引用数。
        if slot==0 or (slot!=0 and self.refs[s.block_table[-1]]>1):
            if not self.free:
                self.oom_count+=1
                raise NoKVSpace('新页或写时复制需要空闲块，但KV页池已满')
            new_page=True
        if new_page:
            bid=self._alloc()
            if slot==0:
                s.block_table.append(bid)
            else:
                cow_old=s.block_table[-1]
                with torch.no_grad():self.data[bid].copy_(self.data[cow_old])
                self._decref(cow_old)
                s.block_table[-1]=bid
                self.cow_count+=1
        else:
            bid=s.block_table[-1]
        s.pending={'token_id':token_id,'slot':slot,'block':bid,
                   'new_page':new_page,'cow_old':cow_old,'written':set()}

    def put_layer(self,sid:str,layer:int,k:torch.Tensor,v:torch.Tensor):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'没有开始token写入')
        p=self.sessions[sid].pending
        require(type(layer) is int and 0<=layer<self.cfg.layers,'层ID非法')
        require(layer not in p['written'],'同一层重复写入')
        expected=(self.cfg.heads,self.head_dim)
        require(k.shape==expected and v.shape==expected and k.dtype==self.dtype and v.dtype==self.dtype,
                'K/V形状或dtype非法')
        require(k.device.type=='cpu' and v.device.type=='cpu','此教学页池只接受CPU K/V')
        require(bool(torch.isfinite(k).all()) and bool(torch.isfinite(v).all()),'K/V有非有限值')
        with torch.no_grad():
            self.data[p['block'],layer,0,p['slot']].copy_(k.detach())
            self.data[p['block'],layer,1,p['slot']].copy_(v.detach())
        p['written'].add(layer)

    def read_layer(self,sid:str,layer:int) -> tuple[torch.Tensor,torch.Tensor]:
        require(sid in self.sessions and self.sessions[sid].pending is not None,'read必须在token事务中')
        s=self.sessions[sid];p=s.pending
        require(layer in p['written'],'只能读取已写入当前token的层')
        length=len(s.tokens)+1
        ks=[];vs=[]
        for index,bid in enumerate(s.block_table):
            n=min(self.block_size,length-index*self.block_size)
            if n<=0:break
            # backing的slot轴在前，需要转成[heads,time,head_dim]。
            ks.append(self.data[bid,layer,0,:n].transpose(0,1))
            vs.append(self.data[bid,layer,1,:n].transpose(0,1))
        require(bool(ks) and sum(x.shape[1] for x in ks)==length,'KV块表与长度不同步')
        return torch.cat(ks,dim=1),torch.cat(vs,dim=1)

    def commit(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可提交的token')
        s=self.sessions[sid];p=s.pending
        require(p['written']==set(range(self.cfg.layers)),'有层尚未写入：禁止提交不完整KV')
        s.tokens.append(p['token_id']);s.pending=None
        self.audit()

    def abort(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可回滚的token')
        s=self.sessions[sid];p=s.pending
        bid=p['block'];slot=p['slot']
        if p['new_page']:
            if p['cow_old'] is not None:
                old=p['cow_old']
                s.block_table[-1]=old
                self.refs[old]+=1
            else:
                require(s.block_table[-1]==bid,'回滚末尾块不一致')
                s.block_table.pop()
            self._decref(bid)
        else:
            # 当前slot仍不属于任何已提交token，清空可能部分写入的K/V。
            with torch.no_grad():self.data[bid,:,:,slot].zero_()
        s.pending=None
        self.audit()

    def release(self,sid):
        require(sid in self.sessions,'重复释放或会话不存在')
        s=self.sessions[sid]
        require(s.pending is None,'请先回滚未完成写入再释放')
        for bid in s.block_table:self._decref(bid)
        del self.sessions[sid]
        self.audit()

    def usage(self) -> dict:
        table_ids=[b for s in self.sessions.values() for b in s.block_table]
        live=self.capacity_blocks-len(self.free)
        # 若block被两个会话引用，其有效token只计物理占用一次。
        filled={}
        for s in self.sessions.values():
            length=len(s.tokens)+(1 if s.pending else 0)
            for i,bid in enumerate(s.block_table):
                filled[bid]=max(filled.get(bid,0),min(self.block_size,length-i*self.block_size))
        occupied=sum(filled.values())
        logical_refs=len(table_ids)
        return {'capacity_blocks':self.capacity_blocks,'block_size':self.block_size,
                'free_blocks':len(self.free),'live_blocks':live,
                'physical_storage_bytes':self.data.numel()*self.data.element_size(),
                'bytes_per_token':self.bytes_per_token,'bytes_per_block':self.bytes_per_block,
                'live_reserved_bytes':live*self.bytes_per_block,
                'physical_used_slots':occupied,'live_content_bytes':occupied*self.bytes_per_token,
                'physical_internal_fragmentation_bytes':(live*self.block_size-occupied)*self.bytes_per_token,
                'logical_token_uses':sum(len(s.tokens) for s in self.sessions.values()),
                'logical_page_references':logical_refs,
                'saved_physical_blocks_by_sharing':logical_refs-live,
                'session_count':len(self.sessions),'peak_live_blocks':self.peak_live_blocks,
                'peak_sessions':self.peak_sessions,'allocation_count':self.allocation_count,
                'cow_count':self.cow_count,'oom_count':self.oom_count,
                'zeroized_releases':self.zeroized_releases}

    def audit(self) -> dict:
        expected=[0]*self.capacity_blocks
        for sid,s in self.sessions.items():
            require(s.version in self.versions,'会话绑定未知版本')
            require(len(s.block_table)==math.ceil((len(s.tokens)+(1 if s.pending else 0))/self.block_size),
                    '块表页数与有效上下文长度不一致')
            require(len(set(s.block_table))==len(s.block_table),'同一会话不能重复引用物理块')
            for bid in s.block_table:
                require(type(bid) is int and 0<=bid<self.capacity_blocks,'块表有非法物理页')
                expected[bid]+=1
        require(expected==self.refs,'物理块引用计数不等于块表的真实引用')
        require(set(self.free)=={i for i,r in enumerate(self.refs) if r==0},'free list/引用计数不一致')
        require(len(self.free)==len(set(self.free)),'空闲页重复入队')
        require(all(not bool(self.data[i].count_nonzero()) for i in self.free),
                '空闲块包含未清除的KV数据')
        u=self.usage()
        require(u['physical_internal_fragmentation_bytes']>=0,'内部碎片负数')
        require(u['live_content_bytes']<=u['live_reserved_bytes']<=u['physical_storage_bytes'],
                '实际/保留字节账目不守恒')
        return u

In [7]:
cfg = demo_models['v0001'].cfg
probe_pool = PagedKVPool(12, 4, cfg, set(demo_models))
print("物理Tensor形状:", tuple(probe_pool.data.shape))
print("每token真实K/V字节:", probe_pool.bytes_per_token)
print("每页真实K/V字节:", probe_pool.bytes_per_block)
print("空池账本:", probe_pool.audit())

物理Tensor形状: (12, 2, 2, 4, 4, 8)
每token真实K/V字节: 512
每页真实K/V字节: 2048
空池账本: {'capacity_blocks': 12, 'block_size': 4, 'free_blocks': 12, 'live_blocks': 0, 'physical_storage_bytes': 24576, 'bytes_per_token': 512, 'bytes_per_block': 2048, 'live_reserved_bytes': 0, 'physical_used_slots': 0, 'live_content_bytes': 0, 'physical_internal_fragmentation_bytes': 0, 'logical_token_uses': 0, 'logical_page_references': 0, 'saved_physical_blocks_by_sharing': 0, 'session_count': 0, 'peak_live_blocks': 0, 'peak_sessions': 0, 'allocation_count': 0, 'cow_count': 0, 'oom_count': 0, 'zeroized_releases': 0}


## 4. 数学上的缓存账本：不是GPU实测

本课 `layers=2, heads=4, head_dim=8, fp32`，因此每新增一个输入 token 需要 **512 字节** K/V，每页四个 token，即 **2048 字节**。

假设一个**其他模型配置**是 32 层、8 个 KV 头、head_dim=128、bf16：一个 token 的 KV 是 **128 KiB**，8192 token 的**理论 K/V**是 **1 GiB**。该计算**不代表本课在 3060 上加载或运行过这种模型**。

单个序列长度 n 需要 `ceil(n/block_size)` 页，只有最后一页可能出现内部碎片。例如 block_size=4，n=9 需要3页，其中3槽没有有效 token。共享前缀时，还要区分“逻辑页引用次数”和“唯一物理页数”。

In [8]:
# %% 09 显存理论账本、参考练习与独立核验

def theoretical_kv_budget(layers:int,kv_heads:int,head_dim:int,dtype_bytes:int,
                          context_tokens:int,block_size:int) -> dict:
    """仅KV张量的理论账本。忽略模型权重、logits、激活、PyTorch分配器、元数据和碎片。"""
    for name,value in [('layers',layers),('kv_heads',kv_heads),('head_dim',head_dim),
                       ('dtype_bytes',dtype_bytes),('context_tokens',context_tokens),('block_size',block_size)]:
        require(type(value) is int and value>(-1 if name=='context_tokens' else 0),name+'非法')
    per_token=2*layers*kv_heads*head_dim*dtype_bytes
    pages=(context_tokens+block_size-1)//block_size
    return {'layers':layers,'kv_heads':kv_heads,'head_dim':head_dim,'dtype_bytes':dtype_bytes,
            'tokens':context_tokens,'block_size':block_size,
            'bytes_per_token':per_token,'unpaged_KV_bytes':per_token*context_tokens,
            'pages':pages,'page_bytes':per_token*block_size,
            'paged_live_reserved_bytes':pages*block_size*per_token,
            'tail_waste_bytes':(pages*block_size-context_tokens)*per_token}


def budget_demo() -> dict:
    toy=theoretical_kv_budget(2,4,8,4,37,4)
    gqa=theoretical_kv_budget(32,8,128,2,8192,16)
    mha=theoretical_kv_budget(32,32,128,2,8192,16)
    workloads=[theoretical_kv_budget(2,4,8,4,n,4) for n in (5,8,9)]
    return {'toy_cpu_fp32':toy,'hypothetical_32_layer_gqa_bf16':gqa,
            'hypothetical_same_model_mha_bf16':mha,
            'gqa_vs_mha_kv_bytes_ratio':gqa['unpaged_KV_bytes']/mha['unpaged_KV_bytes'],
            'partial_page_examples':workloads,
            'caveat':'理论KV数据容量，不是RTX3060测试或任何完整模型的实测显存'}

In [9]:
budget = budget_demo()
print("玩具真实CPU张量公式:", budget['toy_cpu_fp32'])
print("假设32层GQA配置:", budget['hypothetical_32_layer_gqa_bf16'])
print("相同query头数下GQA/MHA的KV字节比:", budget['gqa_vs_mha_kv_bytes_ratio'])
print("长度5/8/9的页碎片:", [r['tail_waste_bytes'] for r in budget['partial_page_examples']])

玩具真实CPU张量公式: {'layers': 2, 'kv_heads': 4, 'head_dim': 8, 'dtype_bytes': 4, 'tokens': 37, 'block_size': 4, 'bytes_per_token': 512, 'unpaged_KV_bytes': 18944, 'pages': 10, 'page_bytes': 2048, 'paged_live_reserved_bytes': 20480, 'tail_waste_bytes': 1536}
假设32层GQA配置: {'layers': 32, 'kv_heads': 8, 'head_dim': 128, 'dtype_bytes': 2, 'tokens': 8192, 'block_size': 16, 'bytes_per_token': 131072, 'unpaged_KV_bytes': 1073741824, 'pages': 512, 'page_bytes': 2097152, 'paged_live_reserved_bytes': 1073741824, 'tail_waste_bytes': 0}
相同query头数下GQA/MHA的KV字节比: 0.25
长度5/8/9的页碎片: [1536, 0, 1536]


## 5. 模型不是重算全部历史，而是真的从页表读取历史K/V

本课每个新增 token 的计算：

1. 查当前会话已经处理的 token 长度 t，取绝对位置 embedding `position[t]`。
2. 各层只对这一个新 token 计算 q/k/v。
3. 将新 K/V 写入该层的本次物理槽，再**从所有已提交页中按块表顺序读取历史 K/V**。
4. 当前 q 对 `K_0...K_t` 做一次注意力，再经过 FFN。
5. 所有层写完后提交本次 token；任何中途错误都会 abort 未完成的页事务。

因为当前 query 是整个前缀最后一个位置，所有缓存键都是“过去或当前”；此处可以对单 query 使用全部历史 K/V。**如果一次加入多个 query，这种无 mask 的写法不能直接套用，需要按绝对位置定义因果 mask。**

特别注意 PyTorch `scaled_dot_product_attention` 的矩形 `[L,S]` `is_causal=True` 对齐约定；不能假定它自动知道本次 query 在完整前缀中的偏移。[PyTorch 官方文档](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)

In [10]:
# %% 05 增量模型执行器：Q/K/V真实写入、块表聚合与全上下文比较
class CachedModelRunner:
    """一次处理一个新增token。prefill本课逐token执行，非生产高吞吐chunked prefill。

    同一层先写该token的K/V，再用块表读取全部历史K/V，对单query计算attention。
    该前缀计算和原模型full forward的掩码、位置、LoRA权重完全一致。
    """
    def __init__(self,tok:ByteMergeChatTokenizer,models:dict[str,TinyChatLM],pool:PagedKVPool):
        require(set(models)==pool.versions,'模型版本与池的版本集合不一致')
        require(all(m.cfg==pool.cfg for m in models.values()),'模型配置和池配置不匹配')
        require(pool.dtype==torch.float32,'本课模型采用float32，KV存储dtype必须一致')
        require(all(next(m.parameters()).device.type=='cpu' for m in models.values()),'教学KV池仅支持CPU')
        require(all(not p.requires_grad for m in models.values() for p in m.parameters()),
                'KV缓存仅用于冻结推理模型，不能偷偷反传训练')
        self.tok=tok;self.models=models;self.pool=pool
        self.last_logits:dict[str,torch.Tensor]={}
        self.tokens_processed=0;self.attention_query_key_products=0
        self.full_recompute_query_key_products=0
        self.model_hashes={name:state_hash(m.state_dict()) for name,m in models.items()}

    @torch.inference_mode()
    def step(self,sid:str,token_id:int) -> torch.Tensor:
        require(sid in self.pool.sessions,'cache step会话不存在')
        s=self.pool.sessions[sid]
        require(type(token_id) is int and 0<=token_id<self.pool.cfg.vocab_size,'token ID非法')
        length=len(s.tokens)
        require(length<self.pool.cfg.max_length,'上下文长度已满')
        self.pool.begin(sid,token_id,self.pool.cfg.max_length)
        model=self.models[s.version]
        try:
            ids=torch.tensor([[token_id]],dtype=torch.long)
            pos=torch.tensor([length],dtype=torch.long)
            x=model.embed(ids)+model.position(pos)[None]
            heads=self.pool.cfg.heads;head_dim=self.pool.head_dim
            for index,block in enumerate(model.blocks):
                z=block.norm1(x)
                q=block.attn.q_proj(z).reshape(heads,head_dim)
                k=block.attn.k_proj(z).reshape(heads,head_dim)
                v=block.attn.v_proj(z).reshape(heads,head_dim)
                self.pool.put_layer(sid,index,k,v)
                all_k,all_v=self.pool.read_layer(sid,index)
                # query:[heads,1,d]; keys:[heads,seq,d]；此query位于整个缓存最末尾，
                # 所以其可见范围恰好是全部已存的过去token与当前token。
                scores=(q[:,None,:]@all_k.transpose(-2,-1))/math.sqrt(head_dim)
                weights=torch.softmax(scores,dim=-1)
                value=(weights@all_v).reshape(1,1,self.pool.cfg.dim)
                x=x+block.attn.o_proj(value)
                x=x+block.ff(block.norm2(x))
                self.attention_query_key_products+=heads*(length+1)
                self.full_recompute_query_key_products+=heads*(length+1)**2
            next_logits=model.head(model.norm(x))[0,0].detach().clone()
            require(bool(torch.isfinite(next_logits).all()),'模型输出非有限logits')
            self.pool.commit(sid)
            self.last_logits[sid]=next_logits
            self.tokens_processed+=1
            return next_logits
        except Exception:
            if self.pool.sessions[sid].pending is not None:self.pool.abort(sid)
            raise

    def prefill(self,sid:str,ids:list[int]) -> torch.Tensor:
        require(type(ids) is list and bool(ids),'prefill需要非空token列表')
        require(all(type(i) is int and 0<=i<self.pool.cfg.vocab_size for i in ids),
                'prefill出现非法token')
        require(sid in self.pool.sessions,'prefill会话不存在')
        require(len(self.pool.sessions[sid].tokens)+len(ids)<=self.pool.cfg.max_length,
                'prefill不能越界或静默截断')
        last=None
        for token in ids:last=self.step(sid,token)
        return last

    def open_prefill(self,sid,version,ids):
        self.pool.open(sid,version)
        try:return self.prefill(sid,ids)
        except Exception:
            self.close(sid)
            raise

    def fork(self,parent,child,version):
        require(parent in self.last_logits,'父序列尚未prefill，不能fork')
        self.pool.fork(parent,child,version)
        self.last_logits[child]=self.last_logits[parent].clone()

    def close(self,sid):
        self.pool.release(sid)
        self.last_logits.pop(sid,None)

    @torch.inference_mode()
    def full_last(self,version:str,ids:list[int]) -> torch.Tensor:
        require(version in self.models and bool(ids),'版本或输入非法')
        model=self.models[version]
        inp=torch.tensor([ids],dtype=torch.long)
        mask=torch.ones_like(inp,dtype=torch.bool)
        return model(inp,mask)[0,-1].detach().clone()

    def assert_consistent(self,sid,atol=2e-5):
        require(sid in self.pool.sessions and sid in self.last_logits,'尚无缓存输出')
        s=self.pool.sessions[sid]
        expected=self.full_last(s.version,s.tokens)
        got=self.last_logits[sid]
        max_delta=float((expected-got).abs().max())
        require(torch.allclose(expected,got,atol=atol,rtol=2e-5),
                '真实cache输出与full recompute不一致：检查位置、KV存储或因果掩码')
        return max_delta

In [11]:
# %% 07 确定性演练：真实KV计算、共享前缀、内存压力及不泄漏
@torch.inference_mode()
def equivalence_demo(tok,models,seed=7301):
    cfg=next(iter(models.values())).cfg
    pool=PagedKVPool(90,4,cfg,set(models))
    runner=CachedModelRunner(tok,models,pool)
    examples=["Hi", "Hello world!", "我喜欢Python。", "<im_start> literal", "OK 42"]
    rows=[]
    gen=random.Random(seed)
    for version in sorted(models):
        for i,text in enumerate(examples):
            sid=f'eq_{version}_{i}'
            ids=tok.apply_chat_template([{'role':'system','content':'Reply briefly.'},
                                         {'role':'user','content':text}],add_generation_prompt=True)
            runner.open_prefill(sid,version,ids)
            diffs=[runner.assert_consistent(sid)]
            for j in range(4):
                new_id=tok.byte_offset+ord(gen.choice('abcxy0123'))
                runner.step(sid,new_id)
                diffs.append(runner.assert_consistent(sid))
            rows.append({'id':sid,'version':version,'prompt_tokens':len(ids),
                         'checked_lengths':5,'max_logit_abs_delta':max(diffs),
                         'final_context_tokens':len(pool.sessions[sid].tokens)})
            runner.close(sid)
    usage=pool.audit()
    require(usage['live_blocks']==0,'比对结束后还有缓存')
    return {'cases':rows,'max_logit_abs_delta':max(r['max_logit_abs_delta'] for r in rows),
            'runner_tokens_processed':runner.tokens_processed,
            'cached_attention_products':runner.attention_query_key_products,
            'full_recompute_attention_products_for_same_prefixes':runner.full_recompute_query_key_products,
            'theoretical_matmul_products_ratio':runner.full_recompute_query_key_products/
                                              runner.attention_query_key_products,
            'post_release':usage,
            'note':'仅比较QK分数矩阵元素数；不代表真实毫秒、FLOPs或GPU吞吐提升'}


@torch.inference_mode()
def sharing_demo(tok,models):
    cfg=next(iter(models.values())).cfg
    pool=PagedKVPool(24,4,cfg,set(models))
    runner=CachedModelRunner(tok,models,pool)
    ids=tok.apply_chat_template([{'role':'user','content':'Hello world!'}],
                                add_generation_prompt=True)
    # 保证共享一个非满的尾块，以实际触发写时复制，而不只共享满块。
    if len(ids)%pool.block_size==0:ids.append(tok.byte_offset+ord('x'))
    runner.open_prefill('parent','v0001',ids)
    parent_hash_before=state_hash({'kv':pool.data})
    blocks_before=pool.usage()['live_blocks']
    runner.fork('parent','childA','v0001')
    runner.fork('parent','childB','v0001')
    just_fork=pool.usage()
    require(just_fork['live_blocks']==blocks_before and
            just_fork['logical_page_references']==3*blocks_before,
            'fork没有共享全部已有物理页')
    require(runner.assert_consistent('childA')<2e-5,'fork没有继承完整的下一token logits')
    bad_version=False
    try:runner.fork('parent','bad_child','v0002')
    except LessonError:bad_version=True
    require(bad_version and 'bad_child' not in pool.sessions,'跨版本不得共享KV')
    runner.step('childA',tok.byte_offset+ord('a'))
    after_child=pool.usage()
    require(pool.cow_count==1 and after_child['live_blocks']==blocks_before+1,
            '部分页写入没有触发唯一一次COW')
    parent_diff=runner.assert_consistent('parent')
    child_diff=runner.assert_consistent('childA')
    # 第二个子会话仍共享父尾块，写入也必须COW；两个子会话互不污染。
    runner.step('childB',tok.byte_offset+ord('b'))
    require(pool.cow_count==2,'第二个子会话应该写时复制')
    child2_diff=runner.assert_consistent('childB')
    require(pool.sessions['parent'].tokens==ids,'子会话写入不应更改父上下文')
    runner.close('parent')
    remain=pool.usage()
    require(remain['session_count']==2 and remain['live_blocks']>0,
            '父会话释放不能把仍有引用的子会话一起释放')
    runner.close('childA')
    runner.close('childB')
    final=pool.audit()
    require(final['live_blocks']==0 and not bool(pool.data.count_nonzero()),
            '最后一个会话释放后必须清空所有物理KV页')
    return {'prefix_tokens':len(ids),'prefix_pages':blocks_before,
            'after_fork':just_fork,'after_first_COW':after_child,
            'cow_count':pool.cow_count,'cross_version_rejected':bad_version,
            'max_logit_abs_delta':max(parent_diff,child_diff,child2_diff),
            'parent_weight_unchanged':all(state_hash(m.state_dict())==runner.model_hashes[name]
                                          for name,m in models.items()),
            'remaining_after_parent_release':remain,
            'final':final,
            'note':'共享整个物理页；部分页后续写入会复制，不能在旧页上直接覆盖'}


@torch.inference_mode()
def lifecycle_demo(tok,models):
    cfg=next(iter(models.values())).cfg
    make_ids=lambda text:tuple(tok.apply_chat_template([{'role':'user','content':text}],
                                                        add_generation_prompt=True))
    prompts={key:make_ids(text) for key,text in {
        'A':'Hi','B':'OK','C':'Hello!','D':'This request uses a significantly longer prompt.','E':'42'} .items()}
    initial_pages=sum(math.ceil(len(prompts[key])/4) for key in ('A','B','C'))
    capacity=initial_pages+3  # 3个当前会话各额外允许一个生成时的新页。
    require(math.ceil(len(prompts['D'])/4)>3,'测试D应该不能准入')
    pool=PagedKVPool(capacity,4,cfg,set(models))
    runner=CachedModelRunner(tok,models,pool)
    svc=KVService(runner)
    specs={
        'A':RequestSpec('A','v0001',prompts['A'],max_new_tokens=3),
        'B':RequestSpec('B','v0001',prompts['B'],max_new_tokens=3,cancel_at=1),
        'C':RequestSpec('C','v0002',prompts['C'],max_new_tokens=3,deadline_at=1),
        'D':RequestSpec('D','v0002',prompts['D'],max_new_tokens=3),
        'E':RequestSpec('E','v0002',prompts['E'],max_new_tokens=2),
    }
    for sid in ('A','B','C'):require(svc.admit(specs[sid],0),'前3请求应被接纳')
    for sid,char in [('A','a'),('B','b'),('C','c')]:
        # 强制ASCII只为稳定覆盖生命周期；真实模型的K/V仍按该token计算。
        svc.advance(sid,0,tok.byte_offset+ord(char))
    require(not svc.admit(specs['D'],1),'第4条长提示应触发容量拒绝')
    svc.advance('A',1,tok.eos_token_id)
    svc.advance('B',1)
    svc.advance('C',1)
    require(svc.admit(specs['E'],2),'释放A/B/C后新请求应可准入')
    svc.advance('E',2,tok.byte_offset+ord('x'))
    svc.advance('E',3,tok.byte_offset+ord('y'))
    report=svc.report()
    require(report['terminal_counts']=={
        'cancelled':1,'eos':1,'length':1,'rejected_oom':1,'timeout':1},
        '生命周期终态类型/数量不匹配')
    require(all(state_hash(m.state_dict())==runner.model_hashes[name] for name,m in models.items()),
            '推理演练不应修改模型权重')
    report['prefill_lengths']={k:len(v) for k,v in prompts.items()}
    report['pool_capacity_blocks']=capacity
    report['physical_oom_triggered']=pool.oom_count>=1
    report['real_incremental_tokens_processed']=runner.tokens_processed
    report['forced_decisions_explained']='A/B/C/E生命周期的token由显式脚本选择，K/V由真实模型计算'
    return report


@torch.inference_mode()
def memory_stress_demo(tok,models,seed=7320):
    rng=random.Random(seed);pool=PagedKVPool(16,4,next(iter(models.values())).cfg,set(models))
    runner=CachedModelRunner(tok,models,pool)
    prefix=[tok.bos_token_id]+tok.encode('Hello')
    runner.open_prefill('root','v0001',prefix)
    live={'root'};steps=[]
    for i in range(35):
        if len(live)<=1 or (len(live)<7 and rng.random()<0.6):
            parent=rng.choice(sorted(live));child=f'fork_{i:02d}'
            runner.fork(parent,child,'v0001');live.add(child)
            steps.append({'step':i,'action':'fork','n':len(live)})
        else:
            selected=rng.choice(sorted(live-{'root'}))
            runner.close(selected);live.remove(selected)
            steps.append({'step':i,'action':'release','n':len(live)})
        # 新请求可能因已满而无法追加；错误必须不破坏现存会话。
        target=rng.choice(sorted(live))
        try:
            runner.step(target,tok.byte_offset+ord(rng.choice('abc')))
            require(runner.assert_consistent(target)<2e-5,'随机共享后增量输出不匹配')
        except NoKVSpace:
            pool.audit()
        pool.audit()
    for sid in sorted(live):runner.close(sid)
    final=pool.audit()
    require(final['live_blocks']==0 and not bool(pool.data.count_nonzero()),
            '随机生命周期结束仍有脏KV数据')
    return {'steps':len(steps),'peak_sessions':pool.peak_sessions,
            'peak_live_blocks':pool.peak_live_blocks,'cow_events':pool.cow_count,
            'oom_events':pool.oom_count,'allocations':pool.allocation_count,
            'all_released':final['live_blocks']==0,'final':final,
            'note':'固定种子的模拟会话创建/共享/追加/释放；不是并发竞争测试'}

In [12]:
single_pool = PagedKVPool(20,4,cfg,set(demo_models))
single_runner = CachedModelRunner(tokenizer,demo_models,single_pool)
prefix = tokenizer.apply_chat_template([{'role':'user','content':'Hello'}],add_generation_prompt=True)
print('输入token数:',len(prefix))
single_runner.open_prefill('example','v0001',prefix)
print('prefill后cache/full最大logit误差:',single_runner.assert_consistent('example'))
for token in 'abc':
    single_runner.step('example',tokenizer.byte_offset+ord(token))
    print('新增',repr(token),'后最大误差:',single_runner.assert_consistent('example'))
print('token前向计算数:',single_runner.tokens_processed)
print('缓存池活跃块:',single_pool.usage()['live_blocks'])
single_runner.close('example')
print('释放后活跃块:',single_pool.audit()['live_blocks'])

输入token数: 11
prefill后cache/full最大logit误差: 3.5762786865234375e-07
新增 'a' 后最大误差: 2.980232238769531e-07
新增 'b' 后最大误差: 3.5762786865234375e-07
新增 'c' 后最大误差: 3.5762786865234375e-07
token前向计算数: 14
缓存池活跃块: 4
释放后活跃块: 0


## 6. 原样重算验证：不能只比较最终文字

随机小模型可能最终都输出同一个字符，**生成文字相同不证明 KV 缓存正确**。本课对每个模型版本的五种不同 prompt（包含 Unicode 和字面控制符）、每种四个后续 token，在各位置比较：

```text
不缓存：完整上下文 → Transformer.forward() → 最后一行logits
有缓存：一token递增 → 分页读取历史K/V → 最新logits
```

默认 CPU FP32 的最大绝对差异通常在 `1e-6` 以下。严格条件和容差写在代码中，跨设备/版本不能保证位级一致。

另记录理论 **QK 乘法矩阵元素个数**，但并没有测量真实耗时：本课 prefll 逐 token 进行，Python 循环/聚合开销可能远大于数学上的节省；不能把这个比值冒称推理加速倍数。

In [13]:
equiv = equivalence_demo(tokenizer,demo_models)
print('实际完整比对案例数:',len(equiv['cases']))
print('最大logit误差:',equiv['max_logit_abs_delta'])
print('缓存方案QK元素数:',equiv['cached_attention_products'])
print('全前缀重算QK元素数:',equiv['full_recompute_attention_products_for_same_prefixes'])
print('这仅是QK元素计数比，不是真实延迟倍数:',equiv['theoretical_matmul_products_ratio'])
print('结束后的物理页:',equiv['post_release']['live_blocks'])

实际完整比对案例数: 10
最大logit误差: 4.76837158203125e-07
缓存方案QK元素数: 57120
全前缀重算QK元素数: 1516960
这仅是QK元素计数比，不是真实延迟倍数: 26.557422969187677
结束后的物理页: 0


## 7. 共享前缀：fork只增加引用，不复制整段K/V

两个请求的 prompt 前缀相同且模型权重/模板/位置等身份一致时，可以共享完整物理页。父会话的块表复制给子会话，**物理页refcount增加**，物理块占用不增加。

但最后一页可能没有写满。如果两个请求在同一共享尾页上直接写入不同 token，彼此就会污染：因此写入共享的部分页必须先 Copy-on-Write（COW），包括所有层的K/V。

```text
parent: [0,1,2] ───┐
childA: [0,1,2] ───┼── 共享物理页2，ref=3
childB: [0,1,2] ───┘

childA后续写入 → 分配新页7、复制原页2、更新childA的块表
parent和childB仍然指向原页2，内容完全不变
```

本课仅在**相同版本**的合成 prompt 上主动fork。真实服务还必须检查租户/访问权限、模板、完整前缀身份和复用策略；不能仅因文本相似就跨租户共享。

In [14]:
shared = sharing_demo(tokenizer,demo_models)
print('共享的前缀token数:',shared['prefix_tokens'])
print('原来物理页数:',shared['prefix_pages'])
print('三份逻辑前缀fork后live页数:',shared['after_fork']['live_blocks'])
print('COW实际次数:',shared['cow_count'])
print('跨adapter版本fork被拒绝:',shared['cross_version_rejected'])
print('最大logits误差:',shared['max_logit_abs_delta'])
print('父会话先关闭时，剩余子会话数:',shared['remaining_after_parent_release']['session_count'])
print('全部关闭后live页数:',shared['final']['live_blocks'])

共享的前缀token数: 14
原来物理页数: 4
三份逻辑前缀fork后live页数: 4
COW实际次数: 2
跨adapter版本fork被拒绝: True
最大logits误差: 3.5762786865234375e-07
父会话先关闭时，剩余子会话数: 2
全部关闭后live页数: 0


## 8. 单次写入也要事务化：先预算、再写入、最后提交

每个新 token 写入多层 K/V。不能写完第一层就把 token 计入完整上下文，否则其他层可能还没完成。

本课使用 `begin → put_layer × L → commit`。出错时 `abort`：

- 新页：释放刚领取的页；
- 共享尾页的 COW：把原块表和引用数恢复，再释放副本；
- 独享的旧部分页：把未提交的新增槽清零；
- OOM：在修改块表之前检查空闲容量，不覆盖任何已有 KV。

`pool.audit()` 直接数块表引用，和每个物理块的 refcount 核对；检查 free list 是否重复、空闲页是否仍残留旧KV。

这是同步教学实现，不等于多线程原子事务或完整抢占恢复。

In [15]:
tx_pool = PagedKVPool(2,4,cfg,set(demo_models))
tx_runner = CachedModelRunner(tokenizer,demo_models,tx_pool)
tx_runner.open_prefill('tx','v0001',[tokenizer.bos_token_id])
old_data = state_hash({'physical_kv':tx_pool.data})
tx_pool.begin('tx',tokenizer.byte_offset+ord('a'),cfg.max_length)
# 只写第一层，故意使提交失败。
kv = torch.full((cfg.heads,cfg.dim//cfg.heads),0.25)
tx_pool.put_layer('tx',0,kv,kv)
try:
    tx_pool.commit('tx')
except LessonError as error:
    print('预期拒绝不完整提交:',error)
    tx_pool.abort('tx')
print('回滚后全体KV字节完全一致:',old_data==state_hash({'physical_kv':tx_pool.data}))
print('回滚后cache/full保持一致:',tx_runner.assert_consistent('tx'))
tx_runner.close('tx')
print('全部回收:',tx_pool.audit()['live_blocks']==0)

预期拒绝不完整提交: 有层尚未写入：禁止提交不完整KV
回滚后全体KV字节完全一致: True
回滚后cache/full保持一致: 0.0
全部回收: True


## 9. 同一服务里要释放哪些终态？

延续第72课，请求接纳时绑定模型版本。生存期结束必须从runner移除最新logits，并对每个共享块执行引用计数释放。

本课使用真实模型对 prompt 与后续非EOS token 进行增量K/V计算，但为稳定覆盖终态，生命周期夹具的部分 token **明确由测试脚本指定**，不宣称这些是随机模型自己选出的结果。

固定负载产生五类终态：`eos`、`length`、`cancelled`、`timeout`、`rejected_oom`。另有**真实模型 argmax**的短请求测试。结束后所有会话与live页必须为0。

EOS 已经生成时无需再缓存它供下一次生成；命中长度上限的最后token也无需额外写入下一次不会使用的 K/V。取消/deadline在取下一个token之前检查；中断后不再发送token事件。

In [16]:
# %% 06 请求生命周期：容量拒绝、取消、超时、EOS与长度终止
@dataclass(frozen=True)
class RequestSpec:
    request_id: str
    version: str
    prompt_ids: tuple[int,...]
    max_new_tokens: int=3
    cancel_at: int | None=None
    deadline_at: int | None=None


class KVService:
    """同步、逻辑tick下的请求生命周期；不是HTTP服务器/线程调度/GPU调度器。

    授权version在admit时固定。先检查取消/绝对deadline，才可能进行下一次推理。
    结束时立即释放块引用，防止即使停止生成但KV仍占用的资源泄漏。
    """
    def __init__(self,runner:CachedModelRunner):
        self.runner=runner;self.pool=runner.pool
        self.active:dict[str,dict]={}
        self.done:dict[str,dict]={}
        self.events:list[dict]=[]
        self.current_tick=-1

    def _tick(self,tick):
        require(type(tick) is int and tick>=0 and tick>=self.current_tick,'tick需要非负且不倒退')
        self.current_tick=tick

    def _record(self,tick,sid,event,details=None):
        u=self.pool.usage()
        self.events.append({'tick':tick,'request_id':sid,'event':event,
                            'live_blocks':u['live_blocks'],'free_blocks':u['free_blocks'],
                            'sessions':u['session_count'],**(details or {})})

    def admit(self,spec:RequestSpec,tick:int):
        self._tick(tick)
        require(isinstance(spec,RequestSpec),'必须使用RequestSpec')
        self.pool._sid(spec.request_id)
        require(spec.version in self.pool.versions,'模型版本不存在')
        require(spec.request_id not in self.active and spec.request_id not in self.done,
                '同一请求ID不能重用')
        require(type(spec.prompt_ids) is tuple and bool(spec.prompt_ids) and
                all(type(t) is int and 0<=t<self.pool.cfg.vocab_size for t in spec.prompt_ids),
                '提示token非法')
        require(type(spec.max_new_tokens) is int and spec.max_new_tokens>=1,'生成预算非法')
        require(len(spec.prompt_ids)+spec.max_new_tokens-1<=self.pool.cfg.max_length,
                '上下文与生成上限不可超过模型最大位置')
        for val in (spec.cancel_at,spec.deadline_at):
            require(val is None or (type(val) is int and val>=0),'取消/截止时间非法')
        state={'request_id':spec.request_id,'version':spec.version,'status':'active',
               'max_new_tokens':spec.max_new_tokens,'generated_ids':[],
               'cancel_at':spec.cancel_at,'deadline_at':spec.deadline_at,
               'accepted_at':tick,'prefill_tokens':len(spec.prompt_ids),'finished_at':None}
        try:
            self.runner.open_prefill(spec.request_id,spec.version,list(spec.prompt_ids))
        except NoKVSpace:
            state.update(status='rejected_oom',finished_at=tick)
            self.done[spec.request_id]=state
            self._record(tick,spec.request_id,'reject_oom',{'version':spec.version})
            return False
        self.active[spec.request_id]=state
        self._record(tick,spec.request_id,'admit',{'version':spec.version,
                                                  'prefill_tokens':len(spec.prompt_ids)})
        return True

    def _finish(self,sid,tick,status):
        require(sid in self.active,'终止仅允许活跃会话')
        state=self.active.pop(sid)
        self.runner.close(sid)
        state['status']=status;state['finished_at']=tick
        self.done[sid]=state
        self._record(tick,sid,'terminal',{'status':status,'version':state['version'],
                                          'generated_tokens':len(state['generated_ids'])})

    def advance(self,sid,tick,forced_id=None):
        self._tick(tick)
        require(sid in self.active,'不能对已经结束、拒绝或未接受的请求生成token')
        s=self.active[sid]
        if s['cancel_at'] is not None and tick>=s['cancel_at']:
            self._finish(sid,tick,'cancelled');return None
        if s['deadline_at'] is not None and tick>=s['deadline_at']:
            self._finish(sid,tick,'timeout');return None
        if forced_id is not None:
            require(type(forced_id) is int and 0<=forced_id<self.pool.cfg.vocab_size,'stub token非法')
            require(forced_id==self.runner.tok.eos_token_id or
                    (self.runner.tok.byte_offset<=forced_id<self.runner.tok.byte_offset+128),
                    'stub只能选择EOS或ASCII字符')
            next_id=forced_id;selection='scripted_fixture'
        else:
            tok=self.runner.tok
            choices=[tok.eos_token_id]+[tok.byte_offset+ord(c) for c in 'abcdefghijklmnopqrstuvwxyz0123456789 .,!']
            logits=self.runner.last_logits[sid]
            next_id=choices[int(logits[choices].argmax())]
            selection='model_argmax'
        # EOS生成但不作为下一步上下文去缓存；终止后的最后一个非EOS也无需再算K/V。
        is_terminal=next_id==self.runner.tok.eos_token_id
        hits_length=len(s['generated_ids'])+1>=s['max_new_tokens']
        if not is_terminal and not hits_length:
            try:self.runner.step(sid,next_id)
            except NoKVSpace:
                self._finish(sid,tick,'oom_during_decode')
                return None
        s['generated_ids'].append(next_id)
        self._record(tick,sid,'token',{'token_id':next_id,'selection':selection,
                                      'version':s['version']})
        if is_terminal:self._finish(sid,tick,'eos')
        elif hits_length:self._finish(sid,tick,'length')
        return next_id

    def report(self):
        require(not self.active,'报告前必须处理全部活跃请求，不能掩盖缓存泄漏')
        usage=self.pool.audit()
        require(usage['live_blocks']==0 and usage['session_count']==0,
                '生命周期结束后KV页池尚有占用')
        return {'results':[self.done[k] for k in sorted(self.done)],'events':self.events,
                'final_usage':usage,'terminal_counts':dict(sorted(Counter(
                    r['status'] for r in self.done.values()).items())),
                'weight_hashes':self.runner.model_hashes,
                'scope':'real_tiny_model_prefill_with_explicit_scripted_lifecycle_decisions'}

In [17]:
lifecycle = lifecycle_demo(tokenizer,demo_models)
print('五类终态:',lifecycle['terminal_counts'])
print('CPU真实pool容量blocks:',lifecycle['pool_capacity_blocks'])
print('曾经发生OOM容量拒绝:',lifecycle['physical_oom_triggered'])
print('峰值物理block:',lifecycle['final_usage']['peak_live_blocks'])
print('全部结束剩余block:',lifecycle['final_usage']['live_blocks'])
print('真实增量模型处理token:',lifecycle['real_incremental_tokens_processed'])
print('事件日志前7行:')
for row in lifecycle['events'][:7]:print(row)

五类终态: {'cancelled': 1, 'eos': 1, 'length': 1, 'rejected_oom': 1, 'timeout': 1}
CPU真实pool容量blocks: 10
曾经发生OOM容量拒绝: True
峰值物理block: 10
全部结束剩余block: 0
真实增量模型处理token: 42
事件日志前7行:
{'tick': 0, 'request_id': 'A', 'event': 'admit', 'live_blocks': 2, 'free_blocks': 8, 'sessions': 1, 'version': 'v0001', 'prefill_tokens': 8}
{'tick': 0, 'request_id': 'B', 'event': 'admit', 'live_blocks': 4, 'free_blocks': 6, 'sessions': 2, 'version': 'v0001', 'prefill_tokens': 7}
{'tick': 0, 'request_id': 'C', 'event': 'admit', 'live_blocks': 7, 'free_blocks': 3, 'sessions': 3, 'version': 'v0002', 'prefill_tokens': 12}
{'tick': 0, 'request_id': 'A', 'event': 'token', 'live_blocks': 8, 'free_blocks': 2, 'sessions': 3, 'token_id': 113, 'selection': 'scripted_fixture', 'version': 'v0001'}
{'tick': 0, 'request_id': 'B', 'event': 'token', 'live_blocks': 8, 'free_blocks': 2, 'sessions': 3, 'token_id': 114, 'selection': 'scripted_fixture', 'version': 'v0001'}
{'tick': 0, 'request_id': 'C', 'event': 'token', 'live_blocks

## 10. 容量满了以后，也不能破坏已有用户的缓存

故意创建仅有一个页的池，先把页写满，再尝试新增 token。预期：抛出 `NoKVSpace`，**已提交的 token、块表、物理 K/V 完全不变**。接着对原请求重新做完整前向核对，仍然成功。释放后全部页可重新使用。

真实服务可能按请求优先级抢占、将KV放到CPU、预留块或推迟准入，本课只实现**明确拒绝＋正确回滚**。没有模拟 vLLM 的真实抢占或 swap 策略。

In [18]:
tiny_pool = PagedKVPool(1,2,cfg,set(demo_models))
tiny_runner = CachedModelRunner(tokenizer,demo_models,tiny_pool)
tiny_runner.open_prefill('A','v0001',[tokenizer.bos_token_id,tokenizer.byte_offset+ord('a')])
h = (state_hash({'data':tiny_pool.data}), tiny_pool.refs.copy(), tiny_pool.usage()['live_blocks'])
try:
    tiny_runner.step('A',tokenizer.byte_offset+ord('b'))
except NoKVSpace as error:
    print('预期OOM:',error)
print('缓存内容与引用状态保持不变:',(state_hash({'data':tiny_pool.data}),tiny_pool.refs.copy(),tiny_pool.usage()['live_blocks'])==h)
print('原请求仍可得到一致logits:',tiny_runner.assert_consistent('A'))
tiny_runner.close('A')
print('释放后无残留:',tiny_pool.audit()['live_blocks']==0)

预期OOM: 新页或写时复制需要空闲块，但KV页池已满
缓存内容与引用状态保持不变: True
原请求仍可得到一致logits: 2.384185791015625e-07
释放后无残留: True


## 11. 随机压力：引用数、共享、COW与释放

固定种子产生35次创建/fork/释放事件，间插真实模型 token 计算，随机选择存活会话继续延长。每步都检查引用数、块表和存储，最终强制结束所有剩余会话。

这是确定性顺序压力测试，能抓逻辑泄露；**没有多线程竞争、异步取消、页驱逐或运行中的模型升级**。

In [19]:
stress = memory_stress_demo(tokenizer,demo_models)
print({k:v for k,v in stress.items() if k!='final'})
print('压力结束所有block已释放:',stress['final']['live_blocks']==0)
print('物理页恢复干净:',stress['final']['free_blocks']==stress['final']['capacity_blocks'])

{'steps': 35, 'peak_sessions': 7, 'peak_live_blocks': 14, 'cow_events': 10, 'oom_events': 0, 'allocations': 21, 'all_released': True, 'note': '固定种子的模拟会话创建/共享/追加/释放；不是并发竞争测试'}
压力结束所有block已释放: True
物理页恢复干净: True


## 12. 完整演练：保存独立模型与五份报告

默认会运行真实缓存数值比对、共享/COW、生命周期和压力测试，并保存：

| 文件 | 内容 |
|---|---|
| `models.pt` | 两个完整冻结模型版本及哈希，供可信本地重建 |
| `equivalence.json` | 每模型/提示的cache-vs-full logits误差，理论QK元素数 |
| `sharing.json` | fork、引用数、物理占用与两次COW结果 |
| `lifecycle.json` | EOS/取消/超时/长度/OOM事件与释放日志 |
| `stress.json` | 固定随机负载、峰值页数和最终归零 |
| `budget.json` | FP32/BF16、MHA/GQA的理论账本 |
| `weights.json` / `reload.json` | 不变权重身份与全新对象回放 |
| `manifest.json` / `checks.json` | 环境和检查记录 |

保存的不是活跃KV会话快照，不用于恢复未完成请求。`torch.load(...,weights_only=True)` 只应加载可信本地文件，哈希仅用于识别意外错配。

In [20]:
# %% 10 完整实验保存与可信本地只读重载
def save_models(path:Path,tok,models,seed):
    payload={'version':VERSION,'seed':seed,'cfg':asdict(next(iter(models.values())).cfg),
             'tokenizer':tok.contract(),
             'hashes':{name:state_hash(m.state_dict()) for name,m in models.items()},
             'states':{name:{k:v.detach().cpu().clone() for k,v in m.state_dict().items()}
                       for name,m in models.items()},
             'adapter_rank':4,'adapter_alpha':8.0}
    torch.save(payload,path)


def load_models(path:Path):
    blob=torch.load(path,map_location='cpu',weights_only=True)
    require(blob['version']==VERSION,'版本不匹配')
    tok,models=build_models(blob['seed'])
    require(blob['tokenizer']==tok.contract(),'tokenizer/template契约不匹配')
    cfg=ToyConfig(**blob['cfg']);cfg.validate()
    require(all(model.cfg==cfg for model in models.values()),'模型配置与文件不匹配')
    require(blob['adapter_rank']==4 and blob['adapter_alpha']==8.0,'LoRA结构不一致')
    require(set(blob['states'])==set(models)==set(blob['hashes']),'版本集合缺失或多余')
    for name,model in models.items():
        model.load_state_dict(blob['states'][name],strict=True)
        model.requires_grad_(False).eval()
        require(state_hash(model.state_dict())==blob['hashes'][name],
                '模型权重与所存内容摘要不匹配')
    require(base_hash(models['v0001'])==base_hash(models['v0002']),
            '发布的两个模型基础参数不同，可能串线')
    return tok,models


def fixture_demo(output_root:Path=Path('lesson_73_outputs'),seed:int=7201) -> Path:
    torch.set_num_threads(1)
    require(type(seed) is int and seed>=0,'seed非法')
    output_root=Path(output_root)
    folder=output_root/datetime.now(timezone.utc).strftime('run_%Y%m%d_%H%M%S_%f')
    folder.mkdir(parents=True,exist_ok=False)
    manifest={'version':VERSION,'python':platform.python_version(),
              'torch':str(torch.__version__),'device':'cpu','seed':seed,
              'status':'running','sources':SOURCES,
              'boundaries':['real_CPU_KV_tensors_with_paged_indirection',
                  'tokenwise_prefill_not_parallel_prefill',
                  'not_vLLM_PagedAttention_GPU_kernel',
                  'no_GPU_latency_or_throughput_measurement',
                  'scripted_lifecycle_choices_explicitly_labeled',
                  'no_real_HTTP_or_thread_concurrency',
                  'physical_storage_is_eager_preallocated']}
    write_json(folder/'manifest.json',manifest)
    try:
        tok,models=build_models(seed)
        model_hashes={v:state_hash(m.state_dict()) for v,m in models.items()}
        save_models(folder/'models.pt',tok,models,seed)
        reports={
            'equivalence.json':equivalence_demo(tok,models),
            'sharing.json':sharing_demo(tok,models),
            'lifecycle.json':lifecycle_demo(tok,models),
            'stress.json':memory_stress_demo(tok,models),
            'budget.json':budget_demo(),
        }
        for name,report in reports.items():write_json(folder/name,report)
        for name,model in models.items():
            require(state_hash(model.state_dict())==model_hashes[name],
                    '所有只读推理完成后模型权重不可改变')
            require(all(p.grad is None for p in model.parameters()),
                    '只读推理不应累计梯度')
        write_json(folder/'weights.json',{'hashes':model_hashes,'base_hash':base_hash(models['v0001']),
                                          'config':asdict(models['v0001'].cfg),'tokenizer':tok.contract()})
        replay=reload_run(folder)
        write_json(folder/'reload.json',replay)
        manifest['status']='completed';write_json(folder/'manifest.json',manifest)
        print('完整演练输出:',folder)
        print('最大增量/full logits差异:',reports['equivalence.json']['max_logit_abs_delta'])
        print('共享前缀后写时复制次数:',reports['sharing.json']['cow_count'])
        print('终态分布:',reports['lifecycle.json']['terminal_counts'])
        print('完成后live物理页:',reports['lifecycle.json']['final_usage']['live_blocks'])
        return folder
    except Exception as exc:
        manifest.update(status='failed',error=type(exc).__name__+': '+str(exc))
        write_json(folder/'manifest.json',manifest)
        raise


def reload_run(folder:Path) -> dict:
    folder=Path(folder)
    tok,models=load_models(folder/'models.pt')
    functions={'equivalence.json':equivalence_demo,'sharing.json':sharing_demo,
               'lifecycle.json':lifecycle_demo,'stress.json':memory_stress_demo}
    comparisons={}
    for file,fn in functions.items():
        expect=json.loads((folder/file).read_text(encoding='utf-8'))
        current=fn(tok,models)
        require(current==expect,'重新建立模型与缓存后报告不一致：'+file)
        comparisons[file]={'exact_report_equal':True}
    current_budget=budget_demo()
    require(current_budget==json.loads((folder/'budget.json').read_text(encoding='utf-8')),
            '内存公式重算不一致')
    saved_weights=json.loads((folder/'weights.json').read_text(encoding='utf-8'))
    require(saved_weights['hashes']=={v:state_hash(m.state_dict()) for v,m in models.items()},
            '模型hash与保存的权重报告不符')
    require(saved_weights['tokenizer']==json.loads(canonical(tok.contract())),
            '模型tokenizer契约报告不符')
    return {'reports':comparisons,'budget_exact_equal':True,
            'weights_exact_equal':True,'live_blocks_after_all_replays':0,
            'scope':'fresh_CPU_model_and_empty_pool_replay;not_resume_of_live_KV_sessions'}

In [21]:
RUN_DIRECTORY = None
if RUN_FULL_DEMO:
    RUN_DIRECTORY = fixture_demo(OUTPUT_ROOT, SEED)
else:
    print('完整演练已明确关闭；不会填入伪造运行结果。')

完整演练输出: lesson_73_outputs/run_20261104_111108_005237
最大增量/full logits差异: 4.76837158203125e-07
共享前缀后写时复制次数: 2
终态分布: {'cancelled': 1, 'eos': 1, 'length': 1, 'rejected_oom': 1, 'timeout': 1}
完成后live物理页: 0


## 13. 只读重载：从新的模型和空页池重新开始

重载新建两个模型（验证权重身份），使用**全新的空页池**重新运行四组演练，再对比保存的 JSON 内容，而不是复用内存中的旧会话或老cache对象。

相同CPU与PyTorch环境里，报告预期一致。它不代表跨设备逐位可复现，也不是保存了真实运行中GPU/KV状态的恢复协议。

In [22]:
if RUN_DIRECTORY is not None:
    replay_report = reload_run(RUN_DIRECTORY)
    print('新进程等价回放契约:',replay_report)
else:
    print('没有运行目录，因此不执行磁盘重载。')

新进程等价回放契约: {'reports': {'equivalence.json': {'exact_report_equal': True}, 'sharing.json': {'exact_report_equal': True}, 'lifecycle.json': {'exact_report_equal': True}, 'stress.json': {'exact_report_equal': True}}, 'budget_exact_equal': True, 'weights_exact_equal': True, 'live_blocks_after_all_replays': 0, 'scope': 'fresh_CPU_model_and_empty_pool_replay;not_resume_of_live_KV_sessions'}


## 14. 自动检查：最容易出错的地方

覆盖检查包括：不完整层提交、COW abort、引用计数与free list不一致、OOM时原请求仍有效、跨版本fork禁止、完整与增量logits逐元素一致、EOS与长度区别、取消/超时不再发token、每种终态释放、空闲页清零和固定压力。

检查不能证明产品级性能与并发安全。真正部署还要做多线程/多进程安全、GPU多流同步、取消竞态、tenant身份与前缀权限、OOM驱逐、分块prefill与内存碎片压力测试。

In [23]:
# %% 08 数值与生命周期的固定检查（含失败与故障注入触发点）
class Checks:
    def __init__(self):self.names=[]
    def ok(self,name,condition):
        if not bool(condition):raise AssertionError(name)
        self.names.append(name)
    def equal(self,name,got,want):self.ok(name,got==want)
    def close(self,name,got,want,atol=1e-6):
        a=torch.as_tensor(got,dtype=torch.float64);b=torch.as_tensor(want,dtype=torch.float64)
        self.ok(name,torch.allclose(a,b,atol=atol,rtol=1e-6))
    def rejects(self,name,fn):
        try:fn()
        except (LessonError,ValueError,TypeError):self.names.append(name);return
        raise AssertionError(name+': 应拒绝而未拒绝')


def run_checks() -> dict:
    torch.set_num_threads(1)
    c=Checks()
    seed_all(7301)
    tok,models=build_models()
    cfg=models['v0001'].cfg
    c.equal('模型两个不可变版本',set(models),{'v0001','v0002'})
    c.ok('同一基座身份',base_hash(models['v0001'])==base_hash(models['v0002']))
    c.ok('不同LoRA权重',state_hash(models['v0001'].state_dict())!=state_hash(models['v0002'].state_dict()))
    c.ok('两个版本所有参数冻结',all(not p.requires_grad for m in models.values() for p in m.parameters()))
    c.ok('教学分词器有独立EOS_ID',tok.eos_token_id not in tok.encode('<eos>'))
    c.ok('Unicode字节往返',tok.decode(tok.encode('你好😊'))=='你好😊')
    c.rejects('非法UTF8不静默替换',lambda:tok.decode([tok.byte_offset+0xe4]))
    c.ok('消息正文控制字样不获得控制ID',tok.im_start_token_id not in tok.encode('<im_start>'))
    c.equal('模型为2层',cfg.layers,2)
    c.equal('每头维度',cfg.dim//cfg.heads,8)
    pool=PagedKVPool(12,4,cfg,set(models));runner=CachedModelRunner(tok,models,pool)
    c.equal('正确每token缓存字节',pool.bytes_per_token,512)
    c.equal('正确每block缓存字节',pool.bytes_per_block,2048)
    c.equal('预分配的物理显存模拟字节',pool.usage()['physical_storage_bytes'],24576)
    c.equal('初始live内存为0',pool.usage()['live_reserved_bytes'],0)
    c.equal('初始内部碎片为0',pool.usage()['physical_internal_fragmentation_bytes'],0)
    c.rejects('拒绝容量为0',lambda:PagedKVPool(0,4,cfg,set(models)))
    c.rejects('拒绝每块0个token',lambda:PagedKVPool(2,0,cfg,set(models)))
    c.rejects('拒绝未支持dtype',lambda:PagedKVPool(2,4,cfg,set(models),torch.float16))
    c.rejects('拒绝未知版本建会话',lambda:pool.open('bad','v9999'))
    c.rejects('拒绝空会话ID',lambda:pool.open('','v0001'))
    c.rejects('拒绝非法会话ID',lambda:pool.open('../../xx','v0001'))
    c.rejects('拒绝不存在会话释放',lambda:pool.release('missing'))
    pool.open('A','v0001')
    c.rejects('拒绝重复ID',lambda:pool.open('A','v0001'))
    c.rejects('拒绝读还没写的token',lambda:pool.read_layer('A',0))
    c.rejects('拒绝提交未开始事务',lambda:pool.commit('A'))
    c.rejects('拒绝缺少事务的abort',lambda:pool.abort('A'))
    # 单token事务只写部分层，必须拒绝commit并正确回滚空闲新页。
    pool.begin('A',tok.bos_token_id,cfg.max_length)
    kv=torch.full((cfg.heads,cfg.dim//cfg.heads),0.2)
    pool.put_layer('A',0,kv,kv)
    c.rejects('缺layer不能commit',lambda:pool.commit('A'))
    c.rejects('同一层重复写入拒绝',lambda:pool.put_layer('A',0,kv,kv))
    c.rejects('未写层不能读取',lambda:pool.read_layer('A',1))
    pool.abort('A')
    c.equal('部分新页回滚长度不变',len(pool.sessions['A'].tokens),0)
    c.equal('部分新页回滚还原块表',pool.sessions['A'].block_table,[])
    c.equal('部分新页回滚释放物理页',pool.usage()['live_blocks'],0)
    c.ok('部分写入数据已清零',not bool(pool.data.count_nonzero()))
    c.rejects('拒绝负token编号',lambda:pool.begin('A',-1,cfg.max_length))
    # 真实模型逐token填充、全上下文比对；第一块尚未填满时可abort复用旧尾块。
    runner.step('A',tok.bos_token_id)
    c.ok('实际单token增量前缀一致',runner.assert_consistent('A')<2e-5)
    before_hash=state_hash({'data':pool.data})
    pool.begin('A',tok.byte_offset+ord('a'),cfg.max_length)
    pool.put_layer('A',0,kv,kv)
    pool.abort('A')
    c.equal('独占未提交尾slot回滚整个存储不变',state_hash({'data':pool.data}),before_hash)
    c.ok('abort不改变已提交token',runner.assert_consistent('A')<2e-5)
    c.rejects('请求超出模型上下文',lambda:pool.begin('A',1,len(pool.sessions['A'].tokens)))
    c.rejects('会话未有完整前缀不能用prefill空列表',lambda:runner.prefill('A',[]))
    c.rejects('非法输入token拒绝',lambda:runner.step('A',cfg.vocab_size))
    # 预生成1 token后，fork的尾页尚未满，写时复制和回滚不能破坏父序列。
    runner.fork('A','B','v0001')
    c.equal('共享时只有一个物理页',pool.usage()['live_blocks'],1)
    c.equal('共享时有两条逻辑引用',pool.usage()['logical_page_references'],2)
    c.equal('共享节约1块',pool.usage()['saved_physical_blocks_by_sharing'],1)
    c.rejects('跨版本共享禁止',lambda:runner.fork('A','wrong','v0002'))
    c.rejects('重复子ID禁止',lambda:runner.fork('A','B','v0001'))
    parent_before=state_hash({'data':pool.data})
    pool.begin('B',tok.byte_offset+ord('b'),cfg.max_length)
    c.equal('共享尾页写入触发COW',pool.cow_count,1)
    pool.put_layer('B',0,kv,kv)
    c.rejects('COW部分写入不许commit',lambda:pool.commit('B'))
    pool.abort('B')
    c.equal('COW abort保留父页',pool.usage()['live_blocks'],1)
    c.equal('COW abort恢复父refcount',pool.refs[pool.sessions['A'].block_table[0]],2)
    c.equal('COW abort不污染父KV',state_hash({'data':pool.data}),parent_before)
    c.ok('COW abort后两个前缀仍一致',runner.assert_consistent('B')<2e-5)
    runner.step('B',tok.byte_offset+ord('b'))
    c.equal('实际子页产生独立物理页',pool.usage()['live_blocks'],2)
    c.ok('COW后子输出与full相同',runner.assert_consistent('B')<2e-5)
    c.ok('COW后父输出仍与full相同',runner.assert_consistent('A')<2e-5)
    c.equal('父token未被改写',pool.sessions['A'].tokens,[tok.bos_token_id])
    c.equal('子token延长了1',len(pool.sessions['B'].tokens),2)
    c.equal('COW占用后内部碎片可计数',pool.usage()['physical_internal_fragmentation_bytes'],(8-3)*512)
    c.rejects('不能在未开始事务写layer',lambda:pool.put_layer('B',0,kv,kv))
    c.rejects('不能fork不存在父会话',lambda:runner.fork('no','child','v0001'))
    runner.close('A')
    c.equal('父结束不释放子私有页',pool.usage()['live_blocks'],1)
    c.ok('子还能生成',runner.assert_consistent('B')<2e-5)
    c.rejects('不能重复释放父会话',lambda:runner.close('A'))
    runner.close('B')
    c.equal('所有会话关闭无活块',pool.usage()['live_blocks'],0)
    c.ok('已释放的K/V全部被清零',not bool(pool.data.count_nonzero()))
    c.equal('所有块回空闲池',pool.usage()['free_blocks'],pool.capacity_blocks)
    c.equal('物理refcount全归零',pool.refs,[0]*pool.capacity_blocks)
    # OOM应该是容量失败，不是损坏现有请求。先占满1页，再试图建第2页。
    tight=PagedKVPool(1,2,cfg,set(models));tr=CachedModelRunner(tok,models,tight)
    tr.open_prefill('full','v0001',[tok.bos_token_id,tok.byte_offset+ord('a')])
    h=tensor_hash_safe(tight)
    c.rejects('无空闲页时明确NoKVSpace',lambda:tr.step('full',tok.byte_offset+ord('b')))
    c.equal('OOM前缀仍完整',tight.sessions['full'].tokens,[tok.bos_token_id,tok.byte_offset+ord('a')])
    c.equal('OOM不修改已提交KV',tensor_hash_safe(tight),h)
    c.ok('OOM后原请求仍可核对',tr.assert_consistent('full')<2e-5)
    c.equal('OOM计数有记录',tight.oom_count,1)
    c.rejects('新会话prefill OOM释放失败仍拒绝',lambda:tr.open_prefill('oom','v0001',[tok.bos_token_id]))
    c.ok('prefill失败不遗留新会话','oom' not in tight.sessions)
    tr.close('full');c.equal('OOM后释放清零',tight.usage()['live_blocks'],0)
    # 同一个特定序列的真实增量与full反复核对；包含中文Unicode字节。
    heavy=equivalence_demo(tok,models)
    c.equal('两个版本各五个案例',len(heavy['cases']),10)
    c.ok('真正logits对齐误差很小',heavy['max_logit_abs_delta']<2e-5)
    c.ok('实际只计当前query的scores个数更少',heavy['cached_attention_products']<
         heavy['full_recompute_attention_products_for_same_prefixes'])
    c.ok('比对后全部页释放',heavy['post_release']['live_blocks']==0)
    c.ok('源模型仍无梯度',all(p.grad is None for m in models.values() for p in m.parameters()))
    share=sharing_demo(tok,models)
    c.equal('两子分叉各一次COW',share['cow_count'],2)
    c.ok('fork前缀共享节约至少两个块',share['after_fork']['saved_physical_blocks_by_sharing']>=2)
    c.ok('跨版本fork被拒绝',share['cross_version_rejected'])
    c.ok('释放父后子保持有效',share['remaining_after_parent_release']['session_count']==2)
    c.ok('完全回收共享所有页',share['final']['live_blocks']==0)
    c.ok('共享后模型输出不变化',share['max_logit_abs_delta']<2e-5)
    c.ok('所有模型权重内容不变',share['parent_weight_unchanged'])
    lifecycle=lifecycle_demo(tok,models)
    c.equal('五类终态各一条',lifecycle['terminal_counts'],
            {'cancelled':1,'eos':1,'length':1,'rejected_oom':1,'timeout':1})
    c.ok('生命周期包含真实KV预算拒绝',lifecycle['physical_oom_triggered'])
    c.equal('生命周期终止后无会话',lifecycle['final_usage']['session_count'],0)
    c.equal('生命周期终止后无物理页',lifecycle['final_usage']['live_blocks'],0)
    c.ok('取消和deadline后无额外token事件',all(not(e['event']=='token' and e['tick']>=1 and
          e['request_id'] in {'B','C'}) for e in lifecycle['events']))
    c.ok('EOS是正文外的真实生成token',
         next(r for r in lifecycle['results'] if r['request_id']=='A')['generated_ids'][-1]==tok.eos_token_id)
    c.ok('OOM拒绝不制造正常输出',
         not next(r for r in lifecycle['results'] if r['request_id']=='D')['generated_ids'])
    c.ok('版本在生成事件中始终固定',all(e['version']==
         next(r for r in lifecycle['results'] if r['request_id']==e['request_id'])['version']
         for e in lifecycle['events'] if 'version' in e))
    c.ok('事件不泄露原始提示字段',all('prompt' not in e and 'user_text' not in e for e in lifecycle['events']))
    c.ok('生命周期峰值不能超过分配容量',
         lifecycle['final_usage']['peak_live_blocks']<=lifecycle['pool_capacity_blocks'])
    # 真实模型作argmax（非强制stub）也要处理EOS/长度并释放。
    realpool=PagedKVPool(8,4,cfg,set(models));rr=CachedModelRunner(tok,models,realpool)
    real=KVService(rr)
    ids=tuple(tok.apply_chat_template([{'role':'user','content':'Hi'}],add_generation_prompt=True))
    c.ok('真实模型可以接纳请求',real.admit(RequestSpec('greedy','v0001',ids,max_new_tokens=2),0))
    real.advance('greedy',0)
    if 'greedy' in real.active:real.advance('greedy',1)
    c.ok('真实模型能在budget内终止','greedy' in real.done and real.done['greedy']['status'] in {'eos','length'})
    c.equal('真实模型结束后页回收',real.report()['final_usage']['live_blocks'],0)
    c.rejects('已结束请求不能继续生成',lambda:real.advance('greedy',2))
    c.rejects('同ID再次admit被拒绝',lambda:real.admit(RequestSpec('greedy','v0001',ids),2))
    c.rejects('提示加输出超出位置被拒绝',lambda:real.admit(RequestSpec('too_long','v0001',tuple([tok.bos_token_id]*cfg.max_length),2),3))
    c.rejects('无序倒退tick被拒绝',lambda:real.admit(RequestSpec('time','v0001',ids),0))
    # 固定随机交替共享、追加、释放，检查每一轮引用计数与空闲数据。
    stress=memory_stress_demo(tok,models)
    c.ok('随机压力执行35次',stress['steps']==35)
    c.ok('随机压力最终释放',stress['all_released'])
    c.ok('随机压力有真实COW',stress['cow_events']>0)
    c.ok('随机压力峰值不越容量',stress['peak_live_blocks']<=16)
    # 测试不可变参数/总账与跨版本比较；普通开发者误将容量当显存可用的情形不应出现。
    c.ok('模拟页池物理占用是预分配存储，不等于live占用',
         lifecycle['final_usage']['physical_storage_bytes']>0 and
         lifecycle['final_usage']['live_reserved_bytes']==0)
    c.ok('完整报告明确区别模型和脚本决策',
         '脚本' in lifecycle['forced_decisions_explained'])
    return {'passed':len(c.names),'checks':c.names,
            'scope':'真实CPU KV tensor/增量因果模型与确定性页池；非PagedAttention GPU kernel/吞吐基准'}


def tensor_hash_safe(pool:PagedKVPool):
    """仅用于检查：包括物理页内容、块表与引用状态。"""
    return digest({'data':state_hash({'pages':pool.data}),
                   'refs':pool.refs,'free':pool.free,
                   'tables':{sid:{'version':s.version,'tokens':s.tokens,'blocks':s.block_table}
                             for sid,s in pool.sessions.items()}})

In [24]:
checks = run_checks()
print('固定检查通过:',checks['passed'],'项')
if RUN_DIRECTORY is not None:
    write_json(RUN_DIRECTORY/'checks.json',checks)

固定检查通过: 109 项


## 15. 独立练习：默认关闭检查开关

三道题都可以先手算，再写函数。检查开关默认 False，参考答案放在最后且与练习函数不同名。

In [25]:
def exercise_a(fn:Callable) -> dict:
    c=Checks()
    c.equal('A玩具每token512字节',fn(2,4,8,4),512)
    c.equal('A_GQA_BF16每token128KiB',fn(32,8,128,2),131072)
    c.equal('A_从MHA到GQA减小4倍',fn(32,32,128,2)//fn(32,8,128,2),4)
    c.rejects('A拒绝0层',lambda:fn(0,4,8,4))
    c.rejects('A拒绝小数字节非整数',lambda:fn(2,4,8,1.5))
    return {'exercise':'A','passed':len(c.names)}


def exercise_b(fn:Callable) -> dict:
    c=Checks()
    cases=[(0,1,4,1),(3,1,4,0),(4,1,4,1),(7,5,4,1),(8,9,4,3)]
    for old,new,B,want in cases:
        c.equal(f'B_{old}+{new}_块大小{B}',fn(old,new,B),want)
    c.rejects('B拒绝负已有长度',lambda:fn(-1,1,4))
    c.rejects('B拒绝零块大小',lambda:fn(1,1,0))
    return {'exercise':'B','passed':len(c.names)}


def exercise_c(fn:Callable) -> dict:
    c=Checks()
    c.equal('C_空序列需要新页',fn(0,4,0,3),'new_page')
    c.equal('C_已经满页需要新页',fn(4,4,1,3),'new_page')
    c.equal('C_独享部分页直接写入',fn(5,4,1,0),'append')
    c.equal('C_共享部分页需要COW',fn(5,4,2,1),'copy_on_write')
    c.equal('C_共享部分页无空页拒绝',fn(5,4,2,0),'oom')
    c.equal('C_满页无空页拒绝',fn(4,4,1,0),'oom')
    c.rejects('C_空序列引用数不能为1',lambda:fn(0,4,1,2))
    c.rejects('C_非空序列引用数不能为0',lambda:fn(3,4,0,2))
    return {'exercise':'C','passed':len(c.names)}

### 练习 A：每个token的KV字节数

实现 `bytes_per_token_ex(layers,kv_heads,head_dim,dtype_bytes)`。需要四个严格正整数，返回 K 与 V 的字节和。用32层、8个KV头、head_dim128、bf16手算验证128 KiB。

In [26]:
def bytes_per_token_ex(layers,kv_heads,head_dim,dtype_bytes):
    raise NotImplementedError('请实现KV每token字节公式')

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    print(exercise_a(bytes_per_token_ex))

### 练习 B：追加一段token需要额外分配多少页？

输入旧长度、要追加的长度和每页token数，返回**新增物理页数量（暂不考虑COW）**。

例如 B=4：旧长度3，再追加1，不需要新页；旧长度4，再追加1，需要新页。不要用 `new_tokens // block_size` 直接算。

In [27]:
def needed_new_pages_ex(old_tokens,append_tokens,block_size):
    raise NotImplementedError('请实现追加所需新增页数')

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    print(exercise_b(needed_new_pages_ex))

### 练习 C：共享部分页写入时的资源判定

输入旧token长度、块大小、**最后一页引用数**和空闲物理页数，返回四种结果之一：`append`、`new_page`、`copy_on_write`、`oom`。

判断顺序：

- 空序列或旧长度刚好为块大小整数倍 → 要新页。
- 还在最后部分页、且引用数大于1 → 要COW。
- 独享部分页 → 直接append。
- 需要新物理页但free=0 → OOM。

禁止非空会话的最后页refcount=0，也禁止空会话还带一页引用。

In [28]:
def cow_action_ex(old_tokens,block_size,last_page_refs,free_pages):
    raise NotImplementedError('请实现物理块分配/COW决策')

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    print(exercise_c(cow_action_ex))

## 16. 完整参考答案

先完成练习并打开各自检查开关，最后再核对答案。下面三个函数与练习区不同名，不会替你实现。

In [29]:
def answer_a(layers,kv_heads,head_dim,dtype_bytes):
    require(all(type(v) is int and v>0 for v in (layers,kv_heads,head_dim,dtype_bytes)),
            '需要四个正整数')
    return 2*layers*kv_heads*head_dim*dtype_bytes


def answer_b(old_tokens,append_tokens,block_size):
    require(all(type(v) is int for v in (old_tokens,append_tokens,block_size)) and
            old_tokens>=0 and append_tokens>=0 and block_size>0,'长度和块大小非法')
    return ((old_tokens+append_tokens+block_size-1)//block_size
            -(old_tokens+block_size-1)//block_size)


def answer_c(old_tokens,block_size,last_page_refs,free_pages):
    require(all(type(v) is int for v in (old_tokens,block_size,last_page_refs,free_pages))
            and old_tokens>=0 and block_size>0 and free_pages>=0,'输入非法')
    require((old_tokens==0 and last_page_refs==0) or (old_tokens>0 and last_page_refs>=1),
            '最后一页引用数与序列是否为空不符')
    need_new=old_tokens%block_size==0
    need_cow=not need_new and last_page_refs>1
    if need_new or need_cow:
        if free_pages==0:return 'oom'
        return 'new_page' if need_new else 'copy_on_write'
    return 'append'

In [30]:
print('练习A参考:',exercise_a(answer_a))
print('练习B参考:',exercise_b(answer_b))
print('练习C参考:',exercise_c(answer_c))

练习A参考: {'exercise': 'A', 'passed': 5}
练习B参考: {'exercise': 'B', 'passed': 7}
练习C参考: {'exercise': 'C', 'passed': 8}


## 17. 课程边界与下一课

完成后应能回答：

1. 为什么增量K/V缓存的最后logits要与全上下文因果前向比较，而不只比较文字？
2. 实际预分配的 Tensor、live页容量和逻辑有效token分别是多少字节？
3. 两个会话共享尾页后，为什么不能直接在原页写新的K/V？
4. 释放父会话时，如何保证尚有子会话引用的前缀不被归还空闲池？
5. EOS、长度上限、取消、deadline和OOM，分别何时停止生成、何时回收？
6. 为什么本课的 QK 元素计数不能称作GPU推理加速倍数？

**没有实现的部分：** 真正PagedAttention GPU Kernel、页驱逐/交换、量化KV、滑动窗口、GQA实际模型、跨GPU迁移、线程竞态、多租户安全授权、prefix cache 哈希匹配与异步请求调度。不要直接把CPU同步代码接入真实在线服务。

**下一课预告：第 74 课——Prefix Cache 命中、模型与模板身份、共享授权及失效策略。** 在这一课正确的物理引用/回收契约上增加前缀复用的身份与安全边界。

## 参考资料

1. [vLLM PagedAttention 原论文](https://arxiv.org/abs/2309.06180)：分页、物理块复用与碎片管理的基础动机。
2. [Hugging Face KV Cache 解释](https://huggingface.co/docs/transformers/main/en/cache_explanation)：按层 K/V、增量生成、cache position 与因果mask。
3. [PyTorch scaled dot-product attention](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)：非方形attention mask与causal对齐约定。
4. [PyTorch inference_mode](https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html)：只读前向的上下文与边界。
5. [PyTorch复现边界](https://docs.pytorch.org/docs/stable/notes/randomness.html)：CPU固定种子不等于跨硬件完全一致。

**教材中运行结果来自真实的小型CPU模型与合成请求。** EOS/取消等确定性事件由显式脚本注入，和模型自身argmax输出分开标记。本课没有真实GPU吞吐或显存实测数据。